### The begining of attention
in the following blocks of code we simply just play around with the idea of attention scores.
Attention scores are dot products of one token embedgging (a query) and the rest of the token embeddings.

this dot product is a reflection of how similiar words are to eachother. such as tea and coffee.

In [2]:
import torch
inputs = torch.tensor(
[[0.43, 0.15, 0.89], 
[0.55, 0.87, 0.66], 
[0.57, 0.85, 0.64], 
[0.22, 0.58, 0.33], 
[0.77, 0.25, 0.10], 
[0.05, 0.80, 0.55]] 
)

In [3]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [4]:
normalized_attn_weights = torch.softmax(attn_scores_2, dim=0)
print(normalized_attn_weights)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [5]:
query = inputs[1]
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 +=normalized_attn_weights[i]*x_i
print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


In [6]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [7]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [8]:
attn_weights[0].sum(), attn_weights.sum(dim=-1)

(tensor(1.0000), tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000]))

In [9]:
context_vectors = attn_weights @ inputs
print(context_vectors)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [10]:
context_vec_2, context_vectors[1], torch.equal(context_vec_2,context_vectors) ,torch.allclose(context_vec_2, context_vectors[1])

(tensor([0.4419, 0.6515, 0.5683]),
 tensor([0.4419, 0.6515, 0.5683]),
 False,
 True)

here i found that two context vectors i created werent exactly the same so i printed out to see why!

In [11]:
torch.set_printoptions(precision=13)
print(context_vec_2)
print(context_vectors[1])

tensor([0.4418657124043, 0.6514819860458, 0.5683088302612])
tensor([0.4418657422066, 0.6514819860458, 0.5683088302612])


### This is the begining of real attention implementation. Still a simple approach first till we get to multilayer head attention archeticture

In [12]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [13]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [14]:
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)

tensor([0.4306369423866, 1.4550582170486])


In [15]:
keys = inputs @ W_key
values = inputs @ W_value
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)



keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [16]:
keys_2 = keys[1]
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8523844480515)


### Now a question i've had was why do we even have a weight matrix for query. Why dont we just use the embeddings tokens. 

The conclusion I reached was by using a weight matrix we are giving the model freedom to further tune the activation of important words rather than it being a fixed similiarty output. It will be further tuned to fit the context. 

What all of that means is that we are giving the model more room to model complex relationships with the similarities of tokens!